# 02. Limpieza y Enriquecimiento de Datos (Feature Engineering)

Una vez que entendieron los datos, vamos a prepararlos para que los algoritmos de Machine Learning puedan procesarlos correctamente. En la vida real, lo mejor es empaquetar esto automatizado en Scikit-Learn Pipelines o en scripts de Python (`src/features/build_features.py`). Aquí puedes experimentar con las rutinas de limpieza.

### Instrucciones Generales:
1. **Solvertar problema de calidad:** Solucionar problemas encontrados en el EDA: consistencia, sensibilidad, precisión y completitud.
2. **Codificación Categórica:** El campo `ocean_proximity` es de texto. Conviértelo en variable numérica.
3. **Enriquecimiento (Feature Engineering):** Agrega nuevas métricas útiles.
4. **Escalado de Variables:** Aplica `StandardScaler` para normalizar las variables numéricas.

## 1. Importaciones y Carga del Set de Entrenamiento

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler

%matplotlib inline
plt.rcParams['figure.figsize'] = (12, 5)
sns.set_theme(style='whitegrid')

In [2]:
# Cargamos el set de entrenamiento (nunca tocamos el test set en esta fase)
train = pd.read_csv('../data/interim/train_set.csv')
print(f'Dimensiones del train set: {train.shape}')
train.head()

Dimensiones del train set: (16512, 10)


,longitude,latitude,housing_median_age,total_rooms,total_bedrooms,population,households,median_income,median_house_value,ocean_proximity
0,-122.42,37.80,52.0,3321.0,1115.0,1576.0,1034.0,2.0987,458300.0,NEAR BAY
1,-118.38,34.14,40.0,1965.0,354.0,666.0,357.0,6.0876,483800.0,<1H OCEAN
2,-121.98,38.36,33.0,1083.0,217.0,562.0,203.0,2.4330,101700.0,INLAND
3,-117.11,33.75,17.0,4174.0,851.0,1845.0,780.0,2.2618,96100.0,INLAND
4,-118.15,33.77,36.0,4366.0,1211.0,1912.0,1172.0,3.5292,361800.0,NEAR OCEAN


## 2. Solución de Problemas de Calidad

Del EDA sabemos que `total_bedrooms` tiene valores faltantes (~1%). La estrategia más robusta y sin fuga de datos es imputar con la **mediana del set de entrenamiento**.

In [3]:
# Verificar faltantes
print('Valores faltantes antes de imputar:')
print(train.isnull().sum()[train.isnull().sum() > 0])

Valores faltantes antes de imputar:
total_bedrooms    168
dtype: int64


In [4]:
# Imputar con la mediana (calculada SOLO sobre el train set para evitar data leakage)
bedrooms_median = train['total_bedrooms'].median()
train['total_bedrooms'] = train['total_bedrooms'].fillna(bedrooms_median)

print(f'Mediana usada para imputar total_bedrooms: {bedrooms_median}')
print(f'Faltantes después de imputar: {train["total_bedrooms"].isnull().sum()}')

Mediana usada para imputar total_bedrooms: 434.0
Faltantes después de imputar: 0


## 3. Codificación Categórica — `ocean_proximity`

La variable `ocean_proximity` es **nominal** (sin orden entre categorías como INLAND, NEAR BAY, etc.), por lo tanto usamos **One-Hot Encoding** (`get_dummies`). Una codificación ordinal asignaría un orden artificial que no existe.

In [6]:
# Revisar distribución antes de codificar
print(train['ocean_proximity'].value_counts())

ocean_proximity
<1H OCEAN     7274
INLAND        5301
NEAR OCEAN    2089
NEAR BAY      1846
ISLAND           2
Name: count, dtype: int64


In [7]:
# One-Hot Encoding
train = pd.get_dummies(train, columns=['ocean_proximity'], dtype=float)
print('Columnas después del OHE:')
print([c for c in train.columns if 'ocean' in c])
train.head()

Columnas después del OHE:
['ocean_proximity_<1H OCEAN', 'ocean_proximity_INLAND', 'ocean_proximity_ISLAND', 'ocean_proximity_NEAR BAY', 'ocean_proximity_NEAR OCEAN']


,longitude,latitude,housing_median_age,total_rooms,total_bedrooms,population,households,median_income,median_house_value,ocean_proximity_<1H OCEAN,ocean_proximity_INLAND,ocean_proximity_ISLAND,ocean_proximity_NEAR BAY,ocean_proximity_NEAR OCEAN
0,-122.42,37.80,52.0,3321.0,1115.0,1576.0,1034.0,2.0987,458300.0,0.0,0.0,0.0,1.0,0.0
1,-118.38,34.14,40.0,1965.0,354.0,666.0,357.0,6.0876,483800.0,1.0,0.0,0.0,0.0,0.0
2,-121.98,38.36,33.0,1083.0,217.0,562.0,203.0,2.4330,101700.0,0.0,1.0,0.0,0.0,0.0
3,-117.11,33.75,17.0,4174.0,851.0,1845.0,780.0,2.2618,96100.0,0.0,1.0,0.0,0.0,0.0
4,-118.15,33.77,36.0,4366.0,1211.0,1912.0,1172.0,3.5292,361800.0,0.0,0.0,0.0,0.0,1.0


## 4. Feature Engineering — Nuevas Variables

Las variables crudas como `total_rooms` no son muy informativas por sí solas (un distrito con 10,000 cuartos pero 5,000 hogares es muy diferente a uno con 10,000 cuartos y 100 hogares). Creamos ratios más significativos.

In [9]:
train['rooms_per_household'] = train['total_rooms'] / train['households']
train['population_per_household'] = train['population'] / train['households']
train['bedrooms_per_room'] = train['total_bedrooms'] / train['total_rooms']

print('Nuevas variables creadas. Estadísticas:')
train[['rooms_per_household', 'population_per_household', 'bedrooms_per_room']].describe()

Nuevas variables creadas. Estadísticas:


,rooms_per_household,population_per_household,bedrooms_per_room
count,16512.000000,16512.000000,16512.000000
mean,5.441010,2.995974,0.213727
std,2.574143,4.457373,0.066077
min,0.888889,0.692308,0.037066
25%,4.443636,2.433426,0.175058
50%,5.235573,2.822316,0.203120
75%,6.053843,3.286385,0.239844
max,141.909091,502.461538,2.818182


In [8]:
# Verificar si las nuevas variables tienen mejor correlación con el precio
new_feats = ['rooms_per_household', 'population_per_household', 'bedrooms_per_room']
corr_new = train[new_feats + ['median_house_value']].corr()['median_house_value'].drop('median_house_value')
print('Correlación de nuevas variables con median_house_value:')
print(corr_new.sort_values(ascending=False))

Correlación de nuevas variables con median_house_value:
rooms_per_household         0.143663
population_per_household   -0.038224
bedrooms_per_room          -0.229558
Name: median_house_value, dtype: float64


## 5. Escalado de Variables

Aplicamos `StandardScaler` a las features numéricas. Esto es especialmente importante para modelos basados en gradientes (SGD, Regresión Lineal) y distancias. El scaler se ajusta **solo** sobre el train set.

In [10]:
# Separar features y target
X_train = train.drop('median_house_value', axis=1)
y_train = train['median_house_value']

# Escalar solo columnas numéricas
scaler = StandardScaler()
X_train_scaled = pd.DataFrame(
    scaler.fit_transform(X_train),
    columns=X_train.columns,
    index=X_train.index
)

print('Rango antes de escalar (rooms_per_household):', X_train['rooms_per_household'].min().round(2), '-', X_train['rooms_per_household'].max().round(2))
print('Rango después de escalar (rooms_per_household):', X_train_scaled['rooms_per_household'].min().round(2), '-', X_train_scaled['rooms_per_household'].max().round(2))

Rango antes de escalar (rooms_per_household): 0.89 - 141.91
Rango después de escalar (rooms_per_household): -1.77 - 53.02


## 6. Verificación Final y Conclusiones

*(Completa con tus observaciones)*

In [11]:
print(f'Shape final del set de entrenamiento: {X_train_scaled.shape}')
print(f'Columnas: {list(X_train_scaled.columns)}')
print(f'Faltantes restantes: {X_train_scaled.isnull().sum().sum()}')

Shape final del set de entrenamiento: (16512, 16)
Columnas: ['longitude', 'latitude', 'housing_median_age', 'total_rooms', 'total_bedrooms', 'population', 'households', 'median_income', 'ocean_proximity_<1H OCEAN', 'ocean_proximity_INLAND', 'ocean_proximity_ISLAND', 'ocean_proximity_NEAR BAY', 'ocean_proximity_NEAR OCEAN', 'rooms_per_household', 'population_per_household', 'bedrooms_per_room']
Faltantes restantes: 0
